# BnSentMix — Model 2: Flat Four-Class XLM-R

**Hypothesis H2:** contextual representations from XLM-R improve over the word+character TF-IDF Linear SVM baseline.

This is a controlled flat-classification baseline. It uses ordinary cross-entropy with **no class weights, focal loss, dual-polarity targets, clause module, or spelling augmentation**. Those changes belong to Models 3–6.

Model 1 validation reference: macro-F1 `0.6967`, mixed-F1 `0.5702`, accuracy `0.7240`. The final test remains sealed.

## Fixed protocol and parameters

| Item | Fixed value |
|---|---|
| Dataset | `/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv` |
| Locked split | Model 1 group-disjoint 70/15/15, seed 2026 |
| Backbone | `FacebookAI/xlm-roberta-base` |
| Classification | Four-way softmax, ordinary cross-entropy |
| Maximum length | 128 tokens |
| Effective batch size | 16 × 2 accumulation = 32 |
| Learning rate | 2e-5 |
| Epoch budget | 5, patience 2 |
| Run seeds | 13, 42, 77 |
| Selection metric | Validation macro-F1 |
| Primary reporting | Mean ± SD macro-F1 across seeds |

Use a Kaggle GPU and enable Internet for the first model download. Do not change these parameters after seeing validation results.

In [ ]:
# Core imports
from pathlib import Path
import gc
import hashlib
import inspect
import json
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback,
    Trainer, TrainingArguments, set_seed
)

# Paths and experiment identity
DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model2')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Values copied from the completed Model 1 notebook
EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026

# Label mapping was already confirmed; do not repeat label inspection
LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3

# Model and optimization parameters declared before training
MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
RUN_SEEDS = [13, 42, 77]
REFERENCE_SEED = 42
OPEN_FINAL_TEST = False  # Keep False until every model is frozen.
USE_FP16 = torch.cuda.is_available()

print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Final test open:', OPEN_FINAL_TEST)

In [ ]:
# Reconstruct Model 1's split exactly, then verify its fingerprints.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, 'Raw dataset differs from Model 1.'

# Standardize only the two required columns; no label audit is repeated.
lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)
df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

# Use the identical 20-fold assignment used in Model 1.
splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id
df['split'] = np.where(
    fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test')
)
assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS

split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
assert sha256_file(split_path) == EXPECTED_SPLIT_SHA256, 'Split differs from Model 1.'

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()
display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)

In [ ]:
# Tokenize lazily so the notebook does not store a second encoded copy of the corpus.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class SentimentDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.labels = frame['label'].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # Truncation prevents any sequence from exceeding XLM-R's limit.
        encoded = tokenizer(
            self.texts[index], truncation=True, max_length=MAX_LENGTH
        )
        encoded['labels'] = self.labels[index]
        return encoded

train_dataset = SentimentDataset(train_df)
val_dataset = SentimentDataset(val_df)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if USE_FP16 else None
)
print('Training samples:', len(train_dataset), '| Validation samples:', len(val_dataset))

In [ ]:
# Metrics used consistently by Models 1–6.
def calculate_metrics(labels, predictions):
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='macro', zero_division=0
        )),
        'weighted_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='weighted', zero_division=0
        )),
        'mixed_f1': float(f1_score(
            labels == MIXED_LABEL, predictions == MIXED_LABEL, zero_division=0
        )),
    }

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=1)
    return calculate_metrics(labels, predictions)

ID2LABEL = {label: name for label, name in LABEL_NAMES.items()}
LABEL2ID = {name: label for label, name in LABEL_NAMES.items()}

In [ ]:
# Build version-compatible TrainingArguments for Kaggle's installed Transformers version.
def make_training_arguments(seed, output_dir):
    arguments = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type='linear',
        max_grad_norm=1.0,
        save_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        data_seed=seed,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        report_to=[],
    )
    # Newer versions use eval_strategy; older versions use evaluation_strategy.
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    arguments['eval_strategy' if 'eval_strategy' in argument_names else 'evaluation_strategy'] = 'epoch'
    return TrainingArguments(**arguments)

In [ ]:
# Train one independent seed and retain its validation-selected checkpoint.
def train_one_seed(seed):
    seed_dir = MODEL_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)

    # A fresh model is required so each seed has an independent classifier head.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
    )
    trainer = Trainer(
        model=model,
        args=make_training_arguments(seed, seed_dir),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )],
    )
    trainer.train()

    # load_best_model_at_end=True makes this prediction use the best validation checkpoint.
    prediction_output = trainer.predict(val_dataset)
    predictions = np.argmax(prediction_output.predictions, axis=1)
    metrics = calculate_metrics(val_df['label'].to_numpy(), predictions)

    eval_logs = [row for row in trainer.state.log_history if 'eval_macro_f1' in row]
    best_log = max(eval_logs, key=lambda row: row['eval_macro_f1'])
    # Save one canonical copy of the selected model for later final evaluation.
    best_model_dir = seed_dir / 'best_model'
    trainer.save_model(best_model_dir)
    metrics.update({
        'seed': seed,
        'best_epoch': float(best_log['epoch']),
        'validation_loss': float(best_log['eval_loss']),
        'best_checkpoint': str(best_model_dir),
    })

    # Save compact row-level predictions and the complete training log.
    prediction_frame = val_df[['source_row_id', 'group_id', 'label']].copy()
    prediction_frame['prediction'] = predictions
    prediction_frame.to_csv(seed_dir / 'validation_predictions.csv', index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(seed_dir / 'training_log.csv', index=False)

    # Remove temporary Trainer checkpoints after preserving the selected model.
    for checkpoint_dir in seed_dir.glob('checkpoint-*'):
        shutil.rmtree(checkpoint_dir)

    del trainer, model, prediction_output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, predictions

In [ ]:
# Run the same fixed configuration for all predeclared seeds.
validation_rows = []
validation_predictions = {}
for run_seed in RUN_SEEDS:
    print(f'\n===== Training seed {run_seed} =====')
    seed_metrics, seed_predictions = train_one_seed(run_seed)
    validation_rows.append(seed_metrics)
    validation_predictions[run_seed] = seed_predictions

validation_results = pd.DataFrame(validation_rows).sort_values('seed').reset_index(drop=True)
metric_columns = ['accuracy', 'macro_f1', 'weighted_f1', 'mixed_f1']
validation_summary = validation_results[metric_columns].agg(['mean', 'std']).T

validation_results.to_csv(MODEL_DIR / 'validation_seed_results.csv', index=False)
validation_summary.to_csv(MODEL_DIR / 'validation_mean_std.csv')
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')
display(validation_results.drop(columns='best_checkpoint').round(4))
display(validation_summary.round(4))

In [ ]:
# Compare Model 2's mean validation performance with the completed Model 1 result.
model1_metrics = {'accuracy': 0.7240, 'macro_f1': 0.6967, 'mixed_f1': 0.5702}
comparison_rows = []
for metric in ['accuracy', 'macro_f1', 'mixed_f1']:
    model2_mean = float(validation_results[metric].mean())
    comparison_rows.append({
        'metric': metric,
        'model1': model1_metrics[metric],
        'model2_mean': model2_mean,
        'delta_model2_minus_model1': model2_mean - model1_metrics[metric],
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(MODEL_DIR / 'model1_vs_model2_validation.csv', index=False)
display(comparison.round(4))

In [ ]:
# Use the predeclared reference seed for the per-class diagnostic table.
reference_prediction = validation_predictions[REFERENCE_SEED]
reference_report = classification_report(
    val_df['label'], reference_prediction, labels=LABELS,
    target_names=[LABEL_NAMES[label] for label in LABELS],
    output_dict=True, zero_division=0
)
display(pd.DataFrame(reference_report).T.round(4))
display(pd.DataFrame(
    confusion_matrix(val_df['label'], reference_prediction, labels=LABELS),
    index=[f'true_{LABEL_NAMES[label]}' for label in LABELS],
    columns=[f'pred_{LABEL_NAMES[label]}' for label in LABELS],
))

## Final-test gate

Keep `OPEN_FINAL_TEST=False`. The following cell is present only for the final evaluation stage after Models 1–6 and all hyperparameters are frozen. Running the notebook normally prints `Final test remains sealed.`

In [ ]:
# Do not open the test set during model development.
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    test_dataset = SentimentDataset(test_df)
    test_rows = []

    for run_seed in RUN_SEEDS:
        # Load the canonical validation-selected model saved for this seed.
        best_model_dir = MODEL_DIR / f'seed_{run_seed}' / 'best_model'
        if not best_model_dir.exists():
            raise FileNotFoundError(f'No saved model for seed {run_seed}.')
        model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
        inference_args = TrainingArguments(
            output_dir=str(MODEL_DIR / f'test_seed_{run_seed}'),
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            fp16=USE_FP16, dataloader_num_workers=2, report_to=[]
        )
        inference_trainer = Trainer(
            model=model, args=inference_args,
            data_collator=data_collator, compute_metrics=compute_metrics
        )
        output = inference_trainer.predict(test_dataset)
        predictions = np.argmax(output.predictions, axis=1)
        metrics = calculate_metrics(test_df['label'].to_numpy(), predictions)
        test_rows.append({'seed': run_seed, **metrics})

        prediction_frame = test_df[['source_row_id', 'group_id', 'label']].copy()
        prediction_frame['prediction'] = predictions
        prediction_frame.to_csv(
            MODEL_DIR / f'FINAL_test_predictions_seed_{run_seed}.csv', index=False
        )
        del inference_trainer, model, output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.DataFrame(test_rows)
    final_summary = final_results[metric_columns].agg(['mean', 'std']).T
    final_results.to_csv(MODEL_DIR / 'FINAL_test_seed_results.csv', index=False)
    final_summary.to_csv(MODEL_DIR / 'FINAL_test_mean_std.csv')
    display(final_results.round(4))
    display(final_summary.round(4))